<a href="https://colab.research.google.com/github/K-1610/PM-TuRi2-Preprocessing-KamalulIman/blob/main/PM_P4_KamalulIman_2488010054.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TuRi 2

## Langkah 1 — Menangani nilai hilang (imputasi median/modus).

In [179]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = {
    'usia': [25, np.nan, 30, 45, np.nan, 22, 50, 35, 28, 40],
    'pendapatan': [5000000, 7000000, np.nan, 12000000, 8000000, np.nan, 15000000, 9000000, 6500000, 10500000],
    'pendidikan': ['SMA', 'S1', 'S2', 'S1', 'SMA', 'S1', 'S2', 'S1', 'SMA', 'S2'],
    'kota': ['Jakarta', 'Bandung', 'Jakarta', 'Surabaya', 'Bandung', 'Jakarta', 'Surabaya', 'Bandung', 'Jakarta', 'Surabaya'],
    'membeli': ['Tidak', 'Ya', 'Ya', 'Ya', 'Tidak', 'Tidak', 'Ya', 'Ya', 'Tidak', 'Ya']
}
df = pd.DataFrame(data)
display(df.head())

,usia,pendapatan,pendidikan,kota,membeli
0,25.0,5000000.0,SMA,Jakarta,Tidak
1,NaN,7000000.0,S1,Bandung,Ya
2,30.0,NaN,S2,Jakarta,Ya
3,45.0,12000000.0,S1,Surabaya,Ya
4,NaN,8000000.0,SMA,Bandung,Tidak


In [180]:
df['usia'] = df['usia'].fillna(df['usia'].median())
df['pendapatan'] = df['pendapatan'].fillna(df['pendapatan'].median())

## Langkah 2 — Memisahkan fitur (X) dan label (y).

In [181]:
X = df.drop(columns=['membeli'])
y = df['membeli']

## Langkah 3 — Encoding kategorikal (ordinal & one-hot).

In [182]:
X['pendidikan'] = X['pendidikan'].map({'SMA':0,'S1':1,'S2':2})
X = pd.get_dummies(X, columns=['kota'], dtype=int)
y = y.map({'Tidak':0,'Ya':1})

## Langkah 4 — Membagi data (SEBELUM scaling).

In [183]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
X, y, test_size=0.3, random_state=42)

## Langkah 5 — Penskalaan (fit pada latih, transform pada latih & uji).

In [184]:
from sklearn.preprocessing import StandardScaler
num = ['usia','pendapatan','pendidikan']
sc = StandardScaler()
X_train[num] = sc.fit_transform(X_train[num])
X_test[num] = sc.transform(X_test[num])

In [185]:
display(df.head())

,usia,pendapatan,pendidikan,kota,membeli
0,25.0,5000000.0,SMA,Jakarta,Tidak
1,32.5,7000000.0,S1,Bandung,Ya
2,30.0,8500000.0,S2,Jakarta,Ya
3,45.0,12000000.0,S1,Surabaya,Ya
4,32.5,8000000.0,SMA,Bandung,Tidak


## Langkah 6 — Verifikasi: tidak ada nilai hilang, semua numerik, skala seragam.

### Latihan Mandiri

1. Ganti imputasi usia menjadi mean; bandingkan hasilnya.

Menggunakan mean akan sangat dipengaruhi oleh nilai ekstrem atau outlier pada kolom usia. Jika terdapat nilai yang sangat tinggi (misal usia 100 tahun), nilai mean akan tergeser jauh, sehingga kurang representatif. Menggunakan median biasanya lebih tahan atau stabil terhadap outlier

2. Gunakan MinMaxScaler (0–1) sebagai ganti StandardScaler; tampilkan describe().

In [186]:
from sklearn.preprocessing import MinMaxScaler
sc_minmax = MinMaxScaler()
X_train[num] = sc_minmax.fit_transform(X_train[num])
print(X_train[num].describe())

           usia  pendapatan  pendidikan
count  7.000000    7.000000    7.000000
mean   0.471429    0.471429    0.571429
std    0.349830    0.318665    0.449868
min    0.000000    0.000000    0.000000
25%    0.250000    0.325000    0.250000
50%    0.400000    0.400000    0.500000
75%    0.700000    0.625000    1.000000
max    1.000000    1.000000    1.000000


Ketika metode describe() dijalankan, Anda akan melihat bahwa pada baris nilai min hasilnya adalah 0.0, dan pada baris nilai max hasilnya adalah 1.0


3. Tambahkan kolom kategorikal baru dan terapkan one-hot encoding.

asumsikan pada data mentah kita menambahkan kolom nominal baru bernama 'pekerjaan'.

In [187]:
df['pekerjaan'] = ['Staf', 'Manajer', 'Direktur', 'Manajer', 'Staf', 'Staf', 'Direktur', 'Manajer', 'Staf', 'Direktur']
display(df.head())

,usia,pendapatan,pendidikan,kota,membeli,pekerjaan
0,25.0,5000000.0,SMA,Jakarta,Tidak,Staf
1,32.5,7000000.0,S1,Bandung,Ya,Manajer
2,30.0,8500000.0,S2,Jakarta,Ya,Direktur
3,45.0,12000000.0,S1,Surabaya,Ya,Manajer
4,32.5,8000000.0,SMA,Bandung,Tidak,Staf


4. Jelaskan mengapa fit_transform pada latih tetapi hanya transform pada uji.

Pemanggilan fit_transform pada data latih bertujuan agar model mempelajari parameter skala (seperti rata-rata, deviasi standar, atau nilai min/max) sekaligus menerapkannya. Sedangkan pada data uji, kita hanya menggunakan .transform() agar parameter yang dipelajari secara eksklusif dari data latih diaplikasikan ke data uji. Hal ini sangat penting untuk mencegah terjadinya kebocoran informasi (data leakage)

### Refleksi

1. Mengapa urutan 'split dulu, baru scaling' penting?

penting karena untuk mencegah terjadinya data leakage (kebocoran data) dari data uji ke data latih. Jika penskalaan fitur dilakukan sebelum data dipisah, informasi parameter statistik pada data uji (seperti nilai rata-rata dan deviasi standar) akan "bocor" dan memengaruhi kalkulasi yang dipakai model untuk belajar.

2. Kapan label encoding dan kapan one-hot encoding?

- Label Encoding digunakan untuk memproses tipe fitur data yang bersifat ordinal atau memiliki tingkatan dan urutan yang jelas (sebagai contoh, tingkatan pendidikan: SMA, S1, S2)
- One-Hot Encoding digunakan untuk memproses tipe data yang bersifat nominal atau tidak memiliki hierarki maupun urutan tingkat (sebagai contoh, asal kota: Jakarta, Bandung, dll)

3. Apa perbedaan normalisasi dan standardisasi?

- Kedua metode ini merupakan bagian dari tahap penskalaan fitur untuk memastikan seragamnya skala
- Normalisasi umumnya mengubah skala fitur ke dalam rentang nilai mutlak tertentu, contohnya direntang nilai 0 hingga 1 (menggunakan teknik MinMaxScaler)
- Normalisasi umumnya mengubah skala fitur ke dalam rentang nilai mutlak tertentu, contohnya direntang nilai 0 hingga 1 (menggunakan teknik MinMaxScaler)